In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override=True)
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [5]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class Contactinfo(BaseModel):
    """contact information for a person"""
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the person")

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    response_format = Contactinfo,
)
result = agent.invoke({
    "messages":[{"role":"user","content":"What is the contact information for John Doe?"}]
})
print(result["structured_response"])


name='John Doe' email='john.doe@example.com' phone='555-123-4567'


In [9]:
from pydantic import BaseModel,Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript"""
    task:str = Field(description="The specific task to be completed")
    assignee:str = Field(description="The person responsible for the task")
    priority:Literal["low","medium","high"] = Field(description="The priority of the task")

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!",
    )
)

result = agent.invoke({
    "messages":[{"role":"user","content":"From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})
print(result["structured_response"])

task='Update the project timeline' assignee='Sarah' priority='high'


In [8]:
from pydantic import BaseModel,Field
from typing import Union 
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ContactInfo(BaseModel):
    name:str = Field(description="Person's name")
    email:str = Field(description="Person's email address")

class EventDeails(BaseModel):
    event_name:str = Field(description="Name of the event")
    date:str = Field(description="Event date")

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    response_format=ToolStrategy(
        Union[ContactInfo,EventDeails],
    )
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})
print(result["structured_response"])

name='John Doe' email='john@email.com'


In [ ]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating of 1.5", gt=0, le=5)
    comment:str = Field(description="Review comment")

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    response_format = ProductRating,
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."
)
result = agent.invoke({
    "messages":[{"role":"user","content":"Parse this:"}]
})
print(result["structured_response"])